            # Round 5: The Final Stretch

            This notebook documents the final `trader.py` submission for the 50
            new Round 5 products plus the Ignith manual Ashflow Alpha allocation.

            ## Final replay summary

            - Combined deterministic replay: **64,300.0 XIRECS**
            - Replay horizon: official-style timestamps `0..99,900`, matching the
              1,000-tick log bundle rather than the exploratory full-day files.
            - Fill model: book-crossing plus public-repo-style passive fills from
              bot trades that would have interacted with our improved quote.
            - Fill count: 95 crossing fills, 880 passive fills, 3,096 total filled units
            - Active algorithmic file: `trader.py`
            - Diagnostics script: `scripts/round5_diagnostics.py`

            | Day | Replay PnL |
|---:|---:|
| 2 | 32,862.0 |
| 3 | 5,310.0 |
| 4 | 26,128.0 |

            ## Top replay contributors

            | Product | Replay PnL |
|---|---:|
| `OXYGEN_SHAKE_EVENING_BREATH` | 25,640.0 |
| `UV_VISOR_ORANGE` | 7,003.5 |
| `SNACKPACK_STRAWBERRY` | 5,617.5 |
| `OXYGEN_SHAKE_CHOCOLATE` | 4,242.0 |
| `SLEEP_POD_NYLON` | 3,917.5 |
| `PANEL_2X4` | 3,670.0 |
| `MICROCHIP_OVAL` | 3,089.0 |
| `TRANSLATOR_ASTRO_BLACK` | 2,783.5 |
| `SLEEP_POD_POLYESTER` | 2,395.5 |
| `SNACKPACK_VANILLA` | 1,706.0 |
| `TRANSLATOR_VOID_BLUE` | 1,461.5 |
| `UV_VISOR_YELLOW` | 1,344.5 |
| `SLEEP_POD_COTTON` | 822.0 |
| `MICROCHIP_SQUARE` | 724.0 |
| `GALAXY_SOUNDS_DARK_MATTER` | 0.0 |
| `GALAXY_SOUNDS_BLACK_HOLES` | 0.0 |

## Public IMC repository lessons used

I reviewed public Prosperity writeups and code repositories before
finalizing the Round 5 shape:

| Source | Applicable lesson |
|---|---|
| [Frankfurt Hedgehogs, Prosperity 3, 2nd globally](https://github.com/TimoDiehm/imc-prosperity-3) | Treat Prosperity as a microstructure game first: identify the fair price, improve inside the spread, and use dashboards/backtests to inspect fills. Their writeup emphasizes WallMid/true-price reasoning, inventory clearing, and bot behavior. |
| [Linear Utility, Prosperity 2, 2nd place](https://github.com/ericcccsliu/imc-prosperity-2) | Build a replay harness, grid search simple parameters, and prefer structural edges over decorative correlations. Their most durable edges came from true-price market making, conversion arbitrage, and spread trades. |
| [jmerle, Prosperity 2, 9th overall](https://github.com/jmerle/imc-prosperity-2) | In Round 5, de-anonymized flow can dominate. Their writeup found named-trader directional signals and also warns that overfit directional products can lose badly. |
| [AlphaBaguette, Prosperity 3](https://github.com/Sylvain-Topeza/imc-prosperity-3) | Combine complementary small edges: adaptive market making, informed flow when available, index/spread logic, and strict position limits. |
| [Prosperity preparation discussion](https://www.reddit.com/r/learnquant/comments/1rvf93p/how_to_actually_compete_and_maybe_win_in_imc/) | The common archetypes are fixed-fair market making, basket/stat-arb spreads, options, location arbitrage, and Round 5 trader-ID flow. |

The Round 5 trade files in this dataset do **not** reveal buyer/seller
IDs; every buyer and seller field is blank. Therefore the prior
"copy the informed trader" trick is not directly available here. The
final algorithm instead applies the reusable parts of those writeups:
fair-price market making, fill-aware parameter selection, and only a
few high-confidence directional overlays.

## Local data findings

The zip contains three historical days: days 2, 3, and 4. Each day has
10,000 timestamps for all 50 products.

Main discoveries:

- The strongest repeatable edge is passive inside-spread market making
  on selected products, but the official fill log showed that some
  sleeves with good simulated spread capture had poor queue quality.
  The current build prunes those products rather than fitting to a
  single path.
- `OXYGEN_SHAKE_CHOCOLATE` and
  `OXYGEN_SHAKE_EVENING_BREATH` have jump-reversion events large
  enough to justify crossing the spread. These are deliberately gated
  by a 30-XIREC one-tick move threshold.
- The earlier `ROBOT_DISHES` jump trigger, snack-pack group overlay,
  and individual pebble makers were removed after the official
  1,000-tick log exposed weak realized execution quality.
- Pebbles still have a near-exact five-product sum around 50,000, but
  the tradable edge is too thin after spread and queue cost.
- Products without robust replay contribution are left idle. Unused
  symbols are better than forced variance.

In [ ]:
import json
from pathlib import Path

diagnostics = json.loads(Path("logs/round5_diagnostics.json").read_text())
diagnostics["combined_pnl"], diagnostics["fills"]

In [ ]:
import pandas as pd

totals = diagnostics["per_product_totals"]
pd.Series(totals).sort_values(ascending=False).head(20).to_frame("replay_pnl")

## Trader implementation

The final `trader.py` is Round 5 only. It does not trade any products
from previous rounds.

Strategy layers:

- **Passive selected makers:** quote one tick better than the best
  displayed bid/ask only when the quote still has positive edge to the
  current book mid after inventory skew.
- **Jump-reversion takers:** cross only after very large one-tick
  moves in the two oxygen products where this paid across replay.
- **No anonymous-flow follower:** buyer/seller IDs in the official log
  are blank except for `SUBMISSION`, and cost-aware flow following was
  negative after spread.
- **Risk controls:** all logic respects the hard 10-unit position
  limit per product, and products without robust evidence are idle.

## Ignith manual strategy

Submit the following Ashflow Alpha manual orders:

| Good | Side | % |
|---|---:|---:|
| Sulfur reactor | Buy | 16% |
| Thermalite core | Buy | 14% |
| Lava cake | Sell | 13% |
| Pyroflex cells | Sell | 11% |
| Magma ink | Buy | 8% |
| Ashes of the Phoenix | Sell | 6% |
| Volcanic incense | Buy | 5% |
| Scoria paste | Buy | 4% |
| Obsidian cutlery | Buy | 3% |

This uses 80% of the manual budget and pays 89,200 XIRECS in fees.
The fee rule makes each product's break-even move equal to its
allocation percentage, so the unused 20% is intentional. It avoids
forcing capital into weaker headlines where the quadratic fee can
overwhelm the news edge.